In [14]:
import sys
!{sys.executable} -m pip install numpy
import sys
!{sys.executable} -m pip install python-sat
import sys
!{sys.executable} -m pip install dwave-neal

In [15]:
import numpy as np
from itertools import product
import random
import sys
from pysat.solvers import Minisat22

np.set_printoptions(
    threshold=sys.maxsize, # 전체 출력
    linewidth=150,        # 한 줄 길이를 넉넉하게
    precision=3,          # 소수점 3자리까지
    suppress=True         # 0.000001을 0.으로 표시
)

In [16]:
# [수정 2.5] 연속 계수 → 이산 계수 (Pelofske 2024, Section 2.1)
#   - 기존: coeff_boundary = 1, np.random.uniform(-1, 1) 연속 균일분포
#   - 수정: lin2 {-1, +1} 또는 lin20 {-1.0, -0.9, ..., 0.9, 1.0} 이산 집합
#   - lin2가 SA에 대해 가장 어려운 문제를 생성 (동일 에너지 상태가 많아 축퇴 증가)
COEFF_LIN2 = [-1, 1]                                              # [수정 2.5]
COEFF_LIN20 = [round(-1 + 0.1 * i, 1) for i in range(21)]        # [수정 2.5]

print("lin2:", COEFF_LIN2)
print("lin20:", COEFF_LIN20)

lin2: [-1, 1]
lin20: [-1.0, -0.9, -0.8, -0.7, -0.6, -0.5, -0.4, -0.3, -0.2, -0.1, 0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]


## gen_random_qubo
n x n 크기의 랜덤 QUBO 생성

[수정 2.5] `coeff_type` 파라미터 추가
- `'lin2'`: {-1, +1} — 가장 어려움 (축퇴 많음)
- `'lin20'`: {-1.0, -0.9, ..., 0.9, 1.0} — 중간
- 기존 `np.random.uniform(-1, 1)` 연속 균일분포 제거

In [17]:
# [수정 2.5] coeff_type 파라미터 추가
#   - 기존: np.random.uniform(-coeff_boundary, coeff_boundary, (n, n)) 연속 균일분포
#   - 수정: 이산 계수 집합에서 랜덤 선택 (논문 방식)
def gen_random_qubo(n, coeff_type='lin2'):                        # [수정 2.5] coeff_type 추가
    coeffs = COEFF_LIN2 if coeff_type == 'lin2' else COEFF_LIN20  # [수정 2.5] 이산 계수 선택
    random_qubo = np.array([[random.choice(coeffs) for _ in range(n)] for _ in range(n)])  # [수정 2.5]
    return np.triu(random_qubo)  # [수정] 상삼각만 유지, 하삼각은 0

In [18]:
print("lin2:")
print(gen_random_qubo(5, 'lin2'))       # {-1, +1}만 나옴
print("\nlin20:")
print(gen_random_qubo(5, 'lin20'))      # {-1.0, -0.9, ..., 1.0} 중에서 나옴

lin2:
[[-1 -1 -1 -1  1]
 [ 0 -1 -1  1 -1]
 [ 0  0 -1  1 -1]
 [ 0  0  0  1  1]
 [ 0  0  0  0  1]]

lin20:
[[-0.2  0.1 -0.5  0.5 -0.4]
 [ 0.   0.1  0.5 -0.1 -0.6]
 [ 0.   0.  -0.7  0.2  0.6]
 [ 0.   0.   0.   1.  -0.5]
 [ 0.   0.   0.   0.  -0.8]]


## find_opt_brute_force
brute force로 최적해, 최적 값, 축퇴도 탐색

In [19]:
# matrix mat를 입력 받아, 최적해, 최적 값, 축퇴도를 출력
# [수정] num_degenerate 반환 추가 — brute force로 동일 에너지 상태 개수를 세서 유일성 검증
def find_opt_brute_force(mat, debug=False):
    n = mat.shape[0]

    best_x = None
    best_val = float('inf')
    num_degenerate = 0  # [수정] 축퇴도 카운트 추가

    for bits in product([0,1], repeat=n):
        x = np.array(bits)
        cur_val = x@mat@x # 스칼라값

        if cur_val < best_val - 1e-12:
            best_val = cur_val
            best_x = x
            num_degenerate = 1
        elif abs(cur_val - best_val) < 1e-12:
            num_degenerate += 1

    if debug: print("opt_x:", best_x, "\nopt_val:", best_val, "\ndegeneracy:", num_degenerate)
        
    return best_x, best_val, num_degenerate

## gen_concatenated_random_qubo
subgraph 균등 분할 → concat하여 큰 QUBO 생성

[수정] 논문 방식으로 균등 분할 (Pelofske 2024, Section 2.2)
- 기존: `min_sub_graph_size`, `max_sub_graph_size`로 랜덤 크기 분할
- 수정: `max_sub_graph_size`만 사용, 균등 분할 (크기 차이 최대 1)
- 논문: "identically sized, unless there are any odd divisions in which case ... different by at most 1 variable"

In [20]:
# [수정] 논문 방식 균등 분할 (Pelofske 2024, Section 2.2)
#   - 기존: min_sub_graph_size ~ max_sub_graph_size 랜덤 크기
#   - 수정: max_sub_graph_size 기준 균등 분할 (크기 차이 최대 1)
#   예) n=10, max_sub_graph_size=3 → k=4개 partition → 크기 [3, 3, 2, 2]
def gen_concatenated_random_qubo(n, max_sub_graph_size, coeff_type='lin2', debug=False):  # [수정] min 제거
    # 균등 분할: k개 partition, 각 크기 차이 최대 1                # [수정]
    k = max(1, -(-n // max_sub_graph_size))  # ceil(n / max_size)  # [수정]

    mat = np.zeros((n, n))
    opt = np.zeros(n)
    cur_size = 0

    for i in range(k):                                             # [수정]
        start = i * n // k                                         # [수정]
        end = (i + 1) * n // k                                     # [수정]
        cur_n = end - start                                        # [수정]

        cur_mat = gen_random_qubo(cur_n, coeff_type)
        cur_opt, _, _ = find_opt_brute_force(cur_mat)

        mat[start:end, start:end] = cur_mat                        # [수정] cur_size → start:end
        opt[start:end] = cur_opt                                   # [수정]

        if debug:
            print(f"  partition {i}: vars [{start}, {end}), size={cur_n}")

    if debug:
        print(f"  총 {k}개 partition, 크기: {[((i+1)*n//k - i*n//k) for i in range(k)]}")
        print()

    return mat, opt

In [21]:
gen_concatenated_random_qubo(10, 5, debug=True) # n=10, max_sub_graph_size=5 → 균등 분할

  partition 0: vars [0, 5), size=5
  partition 1: vars [5, 10), size=5
  총 2개 partition, 크기: [5, 5]



(array([[ 1., -1.,  1., -1., -1.,  0.,  0.,  0.,  0.,  0.],
        [ 0., -1.,  1.,  1., -1.,  0.,  0.,  0.,  0.,  0.],
        [ 0.,  0.,  1., -1.,  1.,  0.,  0.,  0.,  0.,  0.],
        [ 0.,  0.,  0.,  1.,  1.,  0.,  0.,  0.,  0.,  0.],
        [ 0.,  0.,  0.,  0.,  1.,  0.,  0.,  0.,  0.,  0.],
        [ 0.,  0.,  0.,  0.,  0.,  1.,  1., -1.,  1., -1.],
        [ 0.,  0.,  0.,  0.,  0.,  0.,  1., -1.,  1.,  1.],
        [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  1., -1.,  1.],
        [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  1.,  1.],
        [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  1.]]),
 array([1., 1., 0., 0., 1., 0., 0., 0., 0., 0.]))

## posiform_planting
posiform planting (MiniSat uniqueness 검증)

별도의 posiform 전용 행렬(`mat_posiform`)에 항을 쌓아서 반환.
- posiform 계수 = α (posiform_scale). α가 작을수록 SA가 어려움
- `gen_posiform_qubo`에서 `Q_final = Q_random + Q_posiform`으로 결합 (별도 스케일링 없음)

In [22]:
# posiform을 별도 행렬에 쌓아서 반환
#   → gen_posiform_qubo에서 Q_final = Q_random + Q_posiform 으로 결합
#   posiform 계수 = α (posiform_scale). 별도 스케일링 불필요
#
# 논문(Hahn 2023, Section 2.3) 방식:
#   - 3개 wrong tuple 중 1개만 랜덤 선택하여 추가
# MiniSat으로 2-SAT uniqueness 검증:
#   - 유일해질 때까지 clause를 계속 추가 (2-SAT phase transition은 O(n))
# 상삼각 행렬 형식 유지:
#   - off-diagonal 업데이트 시 mat[min(i,j)][max(i,j)]에 접근
def posiform_planting(opt, n, alpha):
    mat_posiform = np.zeros((n, n))
    all_tuples = [(0, 0), (0, 1), (1, 0), (1, 1)]

    # CNF clauses: MiniSat uniqueness 검증에 사용
    clauses_cnf = []
    import math
    # 상한: 2-SAT phase transition은 O(n)이므로 10*n이면 충분 - (논문에 나온건 아니다)
    max_clauses = 1000*n

    def add_posiform_term(i, j):
        """pair (i,j)에 대해 wrong tuple 1개를 랜덤 선택하여 posiform 항 + CNF clause 추가"""
        target_tuple = (int(opt[i]), int(opt[j]))

        # target을 제외한 3개 wrong tuple
        wrong_tuples = []
        for t in all_tuples:
            if t != target_tuple:
                wrong_tuples.append(t)

        wi, wj = random.choice(wrong_tuples)

        # 상삼각 형식 유지: off-diagonal은 항상 mat[작은][큰]에 접근
        lo, hi = min(i, j), max(i, j)

        # posiform 계수 = α
        if wi == 0 and wj == 0:   # (1-xi)(1-xj) * α
            mat_posiform[i][i] -= alpha
            mat_posiform[j][j] -= alpha
            mat_posiform[lo][hi] += alpha
        elif wi == 0 and wj == 1: # (1-xi)xj * α
            mat_posiform[j][j] += alpha
            mat_posiform[lo][hi] -= alpha
        elif wi == 1 and wj == 0: # xi(1-xj) * α
            mat_posiform[i][i] += alpha
            mat_posiform[lo][hi] -= alpha
        else:                     # xixj * α
            mat_posiform[lo][hi] += alpha

        # CNF clause 추가: wrong tuple (wi,wj) 배제
        lit_i = (i + 1) if wi == 0 else -(i + 1)
        lit_j = (j + 1) if wj == 0 else -(j + 1)
        clauses_cnf.append([lit_i, lit_j])

    def check_uniqueness():
        """MiniSat으로 target 외 다른 해가 있는지 확인."""
        with Minisat22() as solver:
            for clause in clauses_cnf:
                solver.add_clause(clause)

            # target 차단: "적어도 하나의 변수가 target과 달라야 한다"
            blocking = []
            for i in range(n):
                if int(opt[i]) == 1:
                    blocking.append(-(i + 1))
                else:
                    blocking.append(i + 1)

            solver.add_clause(blocking)
            return not solver.solve()

    check_interval = max(1, n // 4)

    for step in range(max_clauses):
        i, j = random.sample(range(n), 2)
        add_posiform_term(i, j)

        if (step + 1) % check_interval == 0:
            if check_uniqueness():
                print(f"  [posiform] {step + 1} clauses로 유일성 확보")
                return mat_posiform

    print(f"  [Warning] {max_clauses} clauses 후에도 유일성 미확보")
    return mat_posiform

## gen_posiform_qubo
random QUBO + posiform planting 결합

`Q_final = Q_random + Q_posiform`
- α(posiform_scale)는 posiform 계수로 직접 사용됨
- α가 클수록(1.0) SA가 쉽게 풀 수 있음
- α가 작을수록(0.01) random QUBO가 지배적 → SA가 어려워짐
- 논문 실험값: α = 0.1, 0.01 (0.01이 가장 어려움)

In [23]:
# Q_final = Q_random + Q_posiform
# α(posiform_scale)는 posiform 계수로 직접 사용됨
def gen_posiform_qubo(n, max_sub_graph_size, posiform_scale=0.1, coeff_type='lin2'):
    # 1단계: random QUBO 생성 (block-diagonal, 균등 분할)
    mat_random, opt = gen_concatenated_random_qubo(n, max_sub_graph_size, coeff_type)

    # 2단계: posiform QUBO 생성 (계수 = α)
    mat_posiform = posiform_planting(opt, n, posiform_scale)

    # 3단계: 결합 — Q_final = Q_random + Q_posiform
    mat = mat_random + mat_posiform

    opt_val = opt @ mat @ opt

    print(f"  [에너지 분해] E_random={opt @ mat_random @ opt:.4f}, "
          f"E_posiform={opt @ mat_posiform @ opt:.4f}, "
          f"E_total={opt_val:.4f}")

    return mat, opt, opt_val

In [24]:
# [수정] 검증 루프: gen_posiform_qubo로 생성 → brute force로 GS 검증
chk = True
cnt = 0
while chk and cnt < 100:
    cnt += 1
    if(cnt %50 == 0): print(cnt)

    print(f"instance: {cnt}")
    qubo, opt_init, _ = gen_posiform_qubo(10, 5, posiform_scale=0.1)

    opt, opt_val, deg = find_opt_brute_force(qubo, False)
    if not np.array_equal(opt_init, opt):
        print("[Error] Optimum is changed!!!")
        chk = False
    if deg > 1:
        print(f"[Warning] Degenerate ground state: {deg} solutions (cnt={cnt})")

print("Done!" if chk else "Failed!")

instance: 1
  [posiform] 48 clauses로 유일성 확보
  [에너지 분해] E_random=-6.0000, E_posiform=-2.2000, E_total=-8.2000
instance: 2
  [posiform] 30 clauses로 유일성 확보
  [에너지 분해] E_random=-6.0000, E_posiform=-1.0000, E_total=-7.0000
instance: 3
  [posiform] 18 clauses로 유일성 확보
  [에너지 분해] E_random=-6.0000, E_posiform=-0.7000, E_total=-6.7000
instance: 4
  [posiform] 32 clauses로 유일성 확보
  [에너지 분해] E_random=-8.0000, E_posiform=-0.9000, E_total=-8.9000
instance: 5
  [posiform] 36 clauses로 유일성 확보
  [에너지 분해] E_random=-5.0000, E_posiform=-0.9000, E_total=-5.9000
instance: 6
  [posiform] 36 clauses로 유일성 확보
  [에너지 분해] E_random=-7.0000, E_posiform=-0.8000, E_total=-7.8000
instance: 7
  [posiform] 28 clauses로 유일성 확보
  [에너지 분해] E_random=-8.0000, E_posiform=-0.7000, E_total=-8.7000
instance: 8
  [posiform] 34 clauses로 유일성 확보
  [에너지 분해] E_random=-10.0000, E_posiform=-0.7000, E_total=-10.7000
instance: 9
  [posiform] 42 clauses로 유일성 확보
  [에너지 분해] E_random=-4.0000, E_posiform=-1.2000, E_total=-5.2000
instance: 10
  [p

In [25]:
n = 10          # 변수 개수
max_sub_graph_size = 5  # 균등 분할 (논문 방식) 나누어 떨어지지 않으면 최대 1개 차이

# α: posiform 스케일링 (Pelofske 2024)
# 클수록 SA가 쉽게 풀 수 있음, 작을수록 어려움
# 논문 실험값: 0.1, 0.01
posiform_scale = 0.1

# coeff_type: random QUBO 이산 계수
# 'lin2'  → {-1, +1} — 가장 어려움 (축퇴 많음)
# 'lin20' → {-1.0, -0.9, ..., 1.0} — 중간
coeff_type = 'lin2' # or 'lin20'

qubo, opt_x, opt_val = gen_posiform_qubo(n, max_sub_graph_size, posiform_scale, coeff_type)

print(f"\nposiform planted qubo (α={posiform_scale}, coeff={coeff_type}):\n", qubo)
print()
print("opt_x:", opt_x)
print("opt_val:", opt_val)

  [posiform] 44 clauses로 유일성 확보
  [에너지 분해] E_random=-10.0000, E_posiform=-1.4000, E_total=-11.4000

posiform planted qubo (α=0.1, coeff=lin2):
 [[ 0.8 -0.9 -0.9  1.   1.1  0.  -0.2  0.3  0.   0. ]
 [ 0.   1.   1.  -1.1  1.1  0.1 -0.1  0.   0.1 -0.1]
 [ 0.   0.  -0.9 -1.1  1.  -0.1  0.  -0.3 -0.2  0. ]
 [ 0.   0.   0.  -1.3 -1.   0.   0.   0.1  0.   0.2]
 [ 0.   0.   0.   0.  -1.3  0.   0.   0.   0.  -0.1]
 [ 0.   0.   0.   0.   0.  -0.9 -1.  -1.1 -1.   1.2]
 [ 0.   0.   0.   0.   0.   0.   0.9 -1.  -1.  -1. ]
 [ 0.   0.   0.   0.   0.   0.   0.  -1.  -1.2 -1. ]
 [ 0.   0.   0.   0.   0.   0.   0.   0.   1.   1. ]
 [ 0.   0.   0.   0.   0.   0.   0.   0.   0.   1.2]]

opt_x: [0. 0. 1. 1. 1. 1. 1. 1. 1. 0.]
opt_val: -11.4


## SA 실험: 동일 (R, P) 쌍에서 α 변화에 따른 난이도 비교

각 인스턴스에서:
1. **R** = block-diagonal random QUBO 생성 (`gen_concatenated_random_qubo`)
2. **P** = posiform planting (α=1 단위 스케일, `posiform_planting`)
3. `Q = R + α·P` 를 α = 0, 0.001, 0.01, 0.1 에 대해 각각 SA 실행
4. 동일 (R, P)이므로 α 효과만 순수하게 비교 가능

**판정 기준**: Energy 성공률 (`|SA_energy - target_energy| < tol`)

In [26]:
import neal
import time
import io
import os
import json
import contextlib

# ─── 하이퍼파라미터 ───
num_instances = 1000
num_reads = 100
num_sweeps = 1000
energy_tol = 1e-6

sa_n = 500
sa_max_sub = 10
sa_coeff = 'lin2'
alphas = [0, 0.001, 0.01, 0.1]

# ─── 결과 저장 경로 ───
results_dir = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'hardened_posiform', 'results')
os.makedirs(results_dir, exist_ok=True)
results_path = os.path.join(results_dir,
    f'alpha_sweep_n{sa_n}_sub{sa_max_sub}_{sa_coeff}_inst{num_instances}_sw{num_sweeps}.json')

# ─── numpy 행렬 → dict 변환 (neal 입력 형식) ───
def matrix_to_qubo_dict(mat):
    Q = {}
    n = mat.shape[0]
    for i in range(n):
        for j in range(i, n):
            if mat[i][j] != 0:
                Q[(i, j)] = float(mat[i][j])
    return Q

sampler = neal.SimulatedAnnealingSampler()

# ─── 결과 저장 ───
stats = {a: {'eng_ok': 0, 'bit_ok': 0, 'hd_sum': 0.0, 'total': 0} for a in alphas}
all_results = []  # 인스턴스별 상세 결과

print(f"═══ SA 실험: 동일 (R, P)에서 α sweep ═══")
print(f"  n={sa_n}, max_sub={sa_max_sub}, coeff={sa_coeff}")
print(f"  instances={num_instances}, reads={num_reads}, sweeps={num_sweeps}")
print(f"  alphas={alphas}, energy_tol={energy_tol}")
print(f"  저장 경로: {results_path}")
print()

t0 = time.perf_counter()

for inst in range(num_instances):
    # ── 1. R 생성 (block-diagonal random QUBO) ──
    mat_R, opt = gen_concatenated_random_qubo(sa_n, sa_max_sub, sa_coeff)
    target = ''.join(str(int(x)) for x in opt)
    E_R = float(opt @ mat_R @ opt)

    # ── 2. P 생성 (posiform, 단위 스케일 α=1) ──
    with contextlib.redirect_stdout(io.StringIO()):
        mat_P = posiform_planting(opt, sa_n, 1.0)
    E_P = float(opt @ mat_P @ opt)

    inst_result = {'inst': inst, 'target': target, 'E_R': E_R, 'E_P': E_P}

    # ── 3. 각 α에 대해 Q = R + α·P → SA ──
    for alpha in alphas:
        Q = mat_R + alpha * mat_P
        target_energy = E_R + alpha * E_P

        Q_dict = matrix_to_qubo_dict(Q)
        ss = sampler.sample_qubo(Q_dict, num_reads=num_reads, num_sweeps=num_sweeps)

        eng_ok = 0
        bit_ok = 0
        hd_sum = 0
        best_energy = float('inf')
        for sample, energy, _ in ss.data(['sample', 'energy', 'num_occurrences']):
            found = ''.join(str(sample[k]) for k in range(sa_n))
            hd = sum(1 for a, b in zip(target, found) if a != b)
            hd_sum += hd
            best_energy = min(best_energy, energy)
            if found == target:
                bit_ok += 1
            if abs(energy - target_energy) < energy_tol:
                eng_ok += 1

        stats[alpha]['eng_ok'] += eng_ok
        stats[alpha]['bit_ok'] += bit_ok
        stats[alpha]['hd_sum'] += hd_sum
        stats[alpha]['total'] += num_reads

        akey = str(alpha)
        inst_result[f'a{akey}_eng'] = eng_ok
        inst_result[f'a{akey}_bit'] = bit_ok
        inst_result[f'a{akey}_hd'] = round(hd_sum / num_reads, 2)
        inst_result[f'a{akey}_best_E'] = best_energy
        inst_result[f'a{akey}_target_E'] = target_energy

    all_results.append(inst_result)

    # 진행 상황 출력 (100 인스턴스마다)
    if (inst + 1) % 100 == 0:
        elapsed = time.perf_counter() - t0
        eta = elapsed / (inst + 1) * (num_instances - inst - 1)
        line = f"  [{inst+1:>4}/{num_instances}] {elapsed:>6.0f}s (ETA {eta:.0f}s)"
        for alpha in alphas:
            r = stats[alpha]
            rate = 100 * r['eng_ok'] / r['total']
            line += f"  α={alpha}:{rate:>5.1f}%"
        print(line)

        # 중간 저장 (100 인스턴스마다)
        with open(results_path, 'w') as f:
            json.dump({
                'params': {
                    'n': sa_n, 'max_sub': sa_max_sub, 'coeff': sa_coeff,
                    'num_instances': inst + 1, 'num_reads': num_reads,
                    'num_sweeps': num_sweeps, 'alphas': alphas,
                    'energy_tol': energy_tol, 'status': 'in_progress',
                },
                'stats': {str(a): {k: v for k, v in s.items()} for a, s in stats.items()},
                'instances': all_results,
            }, f)

elapsed = time.perf_counter() - t0

# ─── 최종 저장 ───
with open(results_path, 'w') as f:
    json.dump({
        'params': {
            'n': sa_n, 'max_sub': sa_max_sub, 'coeff': sa_coeff,
            'num_instances': num_instances, 'num_reads': num_reads,
            'num_sweeps': num_sweeps, 'alphas': alphas,
            'energy_tol': energy_tol, 'elapsed_s': round(elapsed, 1),
            'status': 'complete',
        },
        'stats': {str(a): {k: v for k, v in s.items()} for a, s in stats.items()},
        'instances': all_results,
    }, f)

# 최종 요약
print(f"\n{'═' * 90}")
print(f"  총 {num_instances} 인스턴스 × {len(alphas)} α × {num_reads} reads = "
      f"{num_instances * len(alphas) * num_reads:,} SA 샘플 | {elapsed:.1f}s")
print(f"  저장: {results_path}")
print(f"{'═' * 90}")
print(f"{'α':<10} {'Energy 성공률':>16} {'Bit 성공률':>16} {'Avg HD':>10}")
print(f"{'─' * 90}")
for alpha in alphas:
    r = stats[alpha]
    eng_rate = 100 * r['eng_ok'] / r['total']
    bit_rate = 100 * r['bit_ok'] / r['total']
    avg_hd = r['hd_sum'] / num_instances
    print(f"{alpha:<10} {r['eng_ok']:>6}/{r['total']} ({eng_rate:>5.1f}%) "
          f"{r['bit_ok']:>6}/{r['total']} ({bit_rate:>5.1f}%) {avg_hd:>9.1f}")
print(f"{'─' * 90}")

═══ SA 실험: 동일 (R, P)에서 α sweep ═══
  n=500, max_sub=10, coeff=lin2
  instances=1000, reads=100, sweeps=1000
  alphas=[0, 0.001, 0.01, 0.1], energy_tol=1e-06
  저장 경로: /home/yideun/qubo_dataset/hardened_posiform/hardened_posiform/results/alpha_sweep_n500_sub10_lin2_inst1000_sw1000.json

  [ 100/1000]    179s (ETA 1607s)  α=0: 88.8%  α=0.001: 13.7%  α=0.01: 35.3%  α=0.1: 99.8%
  [ 200/1000]    357s (ETA 1428s)  α=0: 89.3%  α=0.001: 13.6%  α=0.01: 34.1%  α=0.1: 99.8%
  [ 300/1000]    536s (ETA 1250s)  α=0: 89.7%  α=0.001: 13.7%  α=0.01: 34.7%  α=0.1: 99.8%
  [ 400/1000]    713s (ETA 1070s)  α=0: 89.3%  α=0.001: 14.6%  α=0.01: 35.3%  α=0.1: 99.7%
  [ 500/1000]    892s (ETA 892s)  α=0: 89.4%  α=0.001: 14.7%  α=0.01: 35.4%  α=0.1: 99.8%
  [ 600/1000]   1068s (ETA 712s)  α=0: 89.7%  α=0.001: 15.3%  α=0.01: 36.1%  α=0.1: 99.8%
  [ 700/1000]   1248s (ETA 535s)  α=0: 89.8%  α=0.001: 14.9%  α=0.01: 35.9%  α=0.1: 99.8%
  [ 800/1000]   1428s (ETA 357s)  α=0: 89.8%  α=0.001: 15.1%  α=0.01: 36.2%  α=0

## SA S-curve: GSP vs Sweeps (논문 Fig 7-8 재현)

동일 (R, P) 쌍에서 **α × sweep 수** 를 동시에 변화시켜 S-curve를 그린다.

- **GSP (Ground-State Probability)**: planted target을 정확히 찾은 비율
- **TTS**: 99% 확률로 GS를 찾는 데 필요한 총 sweep 수
- 논문: α=0.01은 10,000 sweeps에서도 미달, α=0.1은 ~1,000 sweeps에서 100%

In [ ]:
import matplotlib.pyplot as plt
import math

# ─── 하이퍼파라미터 ───
alphas_sc = [0, 0.001, 0.01, 0.1]
sweeps_list = [1, 2, 5, 10, 20, 50, 100, 200, 500, 1000, 2000, 5000, 10000]
sc_n = 500
sc_max_sub = 10
sc_coeff = 'lin2'
sc_num_reads = 100
sc_num_instances = 10   # 논문급 정밀도는 50~100, 빠른 테스트는 5~10

print(f"═══ GSP vs Sweeps S-curve ═══")
print(f"  n={sc_n}, max_sub={sc_max_sub}, coeff={sc_coeff}")
print(f"  alphas={alphas_sc}")
print(f"  sweeps={sweeps_list}")
print(f"  instances={sc_num_instances}, reads/sweep={sc_num_reads}")

# ─── 인스턴스 사전 생성 (R, P 공유) ───
print(f"\n[1/3] 인스턴스 생성 중...")
sc_instances = []
for inst in range(sc_num_instances):
    random.seed(inst * 53)
    mat_R, opt = gen_concatenated_random_qubo(sc_n, sc_max_sub, sc_coeff)
    target_str = ''.join(str(int(x)) for x in opt)
    E_R = float(opt @ mat_R @ opt)

    with contextlib.redirect_stdout(io.StringIO()):
        mat_P = posiform_planting(opt, sc_n, 1.0)
    E_P = float(opt @ mat_P @ opt)

    sc_instances.append({
        'mat_R': mat_R, 'mat_P': mat_P, 'opt': opt,
        'target': target_str, 'E_R': E_R, 'E_P': E_P,
    })
    print(f"  instance {inst+1}/{sc_num_instances}")

# ─── SA sweep 실험 ───
print(f"\n[2/3] SA sweep 실험 중...")
sc_data = {a: {} for a in alphas_sc}

t0_sc = time.perf_counter()
total_runs = len(alphas_sc) * len(sweeps_list) * sc_num_instances
run_count = 0

for alpha in alphas_sc:
    for sw in sweeps_list:
        total_reads = 0
        gs_found = 0
        hd_sum = 0

        for inst_d in sc_instances:
            Q = inst_d['mat_R'] + alpha * inst_d['mat_P']
            Q_dict = matrix_to_qubo_dict(Q)
            target_energy = inst_d['E_R'] + alpha * inst_d['E_P']

            ss = sampler.sample_qubo(Q_dict, num_reads=sc_num_reads, num_sweeps=sw)

            for sample, energy, _ in ss.data(['sample', 'energy', 'num_occurrences']):
                found = ''.join(str(sample[k]) for k in range(sc_n))
                hd = sum(1 for a_ch, b_ch in zip(inst_d['target'], found) if a_ch != b_ch)
                hd_sum += hd
                total_reads += 1
                if found == inst_d['target']:
                    gs_found += 1

            run_count += 1

        gsp = gs_found / total_reads if total_reads > 0 else 0
        hd_avg = hd_sum / total_reads if total_reads > 0 else 0
        sc_data[alpha][sw] = {'gsp': gsp, 'hd_avg': hd_avg}

        elapsed = time.perf_counter() - t0_sc
        eta = elapsed / run_count * (total_runs - run_count)
        print(f"  α={alpha}, sw={sw:>5}: GSP={gsp:>6.3f}, HD={hd_avg:>6.1f} "
              f"[{run_count}/{total_runs}, {elapsed:.0f}s, ETA {eta:.0f}s]")

elapsed_sc = time.perf_counter() - t0_sc
print(f"\n  완료: {elapsed_sc:.1f}s")

# ─── [3/3] 시각화 ───
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

colors = {'0': 'gray', '0.001': 'tab:orange', '0.01': 'tab:red', '0.1': 'tab:blue'}
markers = {'0': 'x', '0.001': 'D', '0.01': 's', '0.1': 'o'}

# (a) GSP vs Sweeps
ax = axes[0]
for alpha in alphas_sc:
    sw_vals = sorted(sc_data[alpha].keys())
    gsp_vals = [sc_data[alpha][sw]['gsp'] for sw in sw_vals]
    ax.semilogx(sw_vals, gsp_vals,
                f'-{markers.get(str(alpha), "o")}',
                color=colors.get(str(alpha), 'black'),
                label=f'α={alpha}', markersize=5, linewidth=1.5)
ax.set_xlabel('Number of SA Sweeps')
ax.set_ylabel('Ground-State Probability (GSP)')
ax.set_title(f'(a) GSP vs Sweeps (n={sc_n}, {sc_coeff})')
ax.set_ylim(-0.05, 1.05)
ax.legend()
ax.grid(True, alpha=0.3)

# (b) HD vs Sweeps
ax = axes[1]
for alpha in alphas_sc:
    sw_vals = sorted(sc_data[alpha].keys())
    hd_vals = [sc_data[alpha][sw]['hd_avg'] for sw in sw_vals]
    ax.semilogx(sw_vals, hd_vals,
                f'-{markers.get(str(alpha), "o")}',
                color=colors.get(str(alpha), 'black'),
                label=f'α={alpha}', markersize=5, linewidth=1.5)
ax.set_xlabel('Number of SA Sweeps')
ax.set_ylabel('Avg Hamming Distance')
ax.set_title(f'(b) HD vs Sweeps')
ax.legend()
ax.grid(True, alpha=0.3)

# (c) TTS vs α (sweep=5000 기준)
ax = axes[2]
tts_sweep = 5000
tts_vals = []
tts_alphas = []
for alpha in alphas_sc:
    if alpha == 0:
        continue
    gsp = sc_data[alpha].get(tts_sweep, {}).get('gsp', 0)
    if 0 < gsp < 1:
        tts = tts_sweep * math.log(1 - 0.99) / math.log(1 - gsp)
        tts_vals.append(tts)
        tts_alphas.append(alpha)
    elif gsp >= 1.0:
        tts_vals.append(tts_sweep)
        tts_alphas.append(alpha)

if tts_vals:
    ax.semilogy(range(len(tts_alphas)), tts_vals, 'o-', markersize=8, linewidth=2)
    ax.set_xticks(range(len(tts_alphas)))
    ax.set_xticklabels([f'α={a}' for a in tts_alphas])
    ax.set_ylabel(f'TTS₉₉ (sweeps, at {tts_sweep} sweeps/read)')
    ax.set_title(f'(c) TTS vs α')
    ax.grid(True, alpha=0.3)

plt.tight_layout()

results_dir_sc = os.path.join(os.path.dirname(os.path.abspath('__file__')),
                               'hardened_posiform', 'results')
os.makedirs(results_dir_sc, exist_ok=True)
fname_sc = f's_curve_n{sc_n}_sub{sc_max_sub}_{sc_coeff}_inst{sc_num_instances}.png'
plt.savefig(os.path.join(results_dir_sc, fname_sc), dpi=150)
plt.show()
print(f"  저장: {os.path.join(results_dir_sc, fname_sc)}")

# ─── 결과 테이블 ───
print(f"\n{'Sweeps':>8}", end='')
for alpha in alphas_sc:
    print(f"  α={alpha:<5} GSP   HD", end='')
print()
print('─' * (8 + 20 * len(alphas_sc)))
for sw in sweeps_list:
    print(f"{sw:>8}", end='')
    for alpha in alphas_sc:
        d = sc_data[alpha][sw]
        print(f"  {d['gsp']:>8.3f} {d['hd_avg']:>5.1f}", end='')
    print()

# JSON 저장
sc_save = {
    'params': {'n': sc_n, 'max_sub': sc_max_sub, 'coeff': sc_coeff,
               'num_instances': sc_num_instances, 'num_reads': sc_num_reads,
               'alphas': alphas_sc, 'sweeps': sweeps_list,
               'elapsed_s': round(elapsed_sc, 1)},
    'curves': {str(a): {str(sw): sc_data[a][sw] for sw in sweeps_list} for a in alphas_sc},
}
json_sc = os.path.join(results_dir_sc, fname_sc.replace('.png', '.json'))
with open(json_sc, 'w') as f:
    json.dump(sc_save, f)
print(f"\n  JSON: {json_sc}")